# Catchment Hydrology, MSc course
## Lecture 2: The runoff ratio of US catchments
*Wouter R. Berghuijs*

This notebook uses **CAMELS** (Newman et al., 2015; Addor et al., 2017): 671 small- to
medium-sized catchments across the contiguous United States (median basin size 336 km²), chosen
to be minimally impacted by human activities and each with roughly 20 years of daily streamflow.
The dataset also has catchment attributes spanning topography, climate, land cover, soil, and
geology — see Addor et al. (2017) for full variable definitions.

CAMELS is a small, carefully curated dataset — a useful contrast to its European counterpart,
[EStreams](https://doi.org/10.1038/s41597-024-03706-1) (Nascimento et al., 2024), which covers
17,130 catchments across 41 countries and is compiled from dozens of heterogeneous national
providers. The [EStreams companion notebook](https://github.com/wberghuijs/EStreams_Lecture_RunoffRatio/blob/main/Lecture_EStreams_RunoffRatio.ipynb)
uses the exact same tools built here, so you can directly compare the US and Europe.

### The runoff ratio

The **runoff ratio** is the fraction of precipitation that leaves a catchment as streamflow:

$$RR = \frac{Q}{P}$$

where $Q$ is mean streamflow and $P$ is mean precipitation. It is one of the simplest and most
informative catchment "fingerprints", and the starting point of the Budyko framework, which
relates the runoff ratio (or its complement, the evaporative index) primarily to aridity.

**Your task:** using the interactive tools below, explore which climate and landscape
characteristics relate most strongly to the runoff ratio of US catchments — and which don't. Work
through the numbered questions as you go; they set up the discussion of the Budyko framework in
this lecture.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import geopandas as gpd
import ipywidgets as widgets
from ipywidgets import interact

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 6)

# ---------------------------------------------------------------------------
# Small helpers shared by every interactive tool below
# (identical to the ones in the EStreams companion notebook)
# ---------------------------------------------------------------------------
def _parse_float(text):
    """Turn a text-box entry into a float, or None if it's blank/invalid ('auto')."""
    try:
        return float(text)
    except (TypeError, ValueError):
        return None


def _apply_limits(ax, xmin='', xmax='', ymin='', ymax=''):
    """Manually override an axis' limits when a value is given; leave auto-scaled otherwise."""
    xmin, xmax, ymin, ymax = (_parse_float(v) for v in (xmin, xmax, ymin, ymax))
    if xmin is not None or xmax is not None:
        cur = ax.get_xlim()
        ax.set_xlim(xmin if xmin is not None else cur[0],
                     xmax if xmax is not None else cur[1])
    if ymin is not None or ymax is not None:
        cur = ax.get_ylim()
        ax.set_ylim(ymin if ymin is not None else cur[0],
                     ymax if ymax is not None else cur[1])


def _fit_and_label(ax, xdata, ydata, fit_type):
    """Fit a curve of the chosen type through (xdata, ydata), draw it, and return its equation
    as a string (or None if no fit could be made)."""
    if fit_type == 'none':
        return None
    x = np.asarray(xdata, dtype=float)
    y = np.asarray(ydata, dtype=float)

    def _poly_str(coeffs, powers):
        # coeffs/powers highest-order first; renders e.g. "2.1x^2 - 0.9x + 1.2" (proper minus signs)
        terms = []
        for coef, power in zip(coeffs, powers):
            label = '' if power == 0 else ('x' if power == 1 else f'x^{power}')
            if not terms:
                terms.append(f'{coef:.3g}{label}')
            else:
                terms.append(f'{"-" if coef < 0 else "+"} {abs(coef):.3g}{label}')
        return ' '.join(terms)

    try:
        if fit_type == 'linear':
            b, a = np.polyfit(x, y, 1)
            xs = np.linspace(x.min(), x.max(), 200)
            ys = b * xs + a
            eq = 'y = ' + _poly_str([b, a], [1, 0])
        elif fit_type == 'quadratic':
            c2, c1, c0 = np.polyfit(x, y, 2)
            xs = np.linspace(x.min(), x.max(), 200)
            ys = c2 * xs**2 + c1 * xs + c0
            eq = 'y = ' + _poly_str([c2, c1, c0], [2, 1, 0])
        elif fit_type == 'cubic':
            c3, c2, c1, c0 = np.polyfit(x, y, 3)
            xs = np.linspace(x.min(), x.max(), 200)
            ys = c3 * xs**3 + c2 * xs**2 + c1 * xs + c0
            eq = 'y = ' + _poly_str([c3, c2, c1, c0], [3, 2, 1, 0])
        elif fit_type == 'exponential':
            mask = y > 0
            b, loga = np.polyfit(x[mask], np.log(y[mask]), 1)
            a = np.exp(loga)
            xs = np.linspace(x[mask].min(), x[mask].max(), 200)
            ys = a * np.exp(b * xs)
            eq = f'y = {a:.3g}*e^({b:.3g}x)'
        elif fit_type == 'logarithmic':
            mask = x > 0
            b, a = np.polyfit(np.log(x[mask]), y[mask], 1)
            xs = np.linspace(x[mask].min(), x[mask].max(), 200)
            ys = b * np.log(xs) + a
            eq = f'y = {b:.3g}*ln(x) + {a:.3g}'
        elif fit_type == 'power':
            mask = (x > 0) & (y > 0)
            b, loga = np.polyfit(np.log(x[mask]), np.log(y[mask]), 1)
            a = np.exp(loga)
            xs = np.linspace(x[mask].min(), x[mask].max(), 200)
            ys = a * xs**b
            eq = f'y = {a:.3g}*x^{b:.3g}'
        else:
            return None
    except Exception:
        return None

    ax.plot(xs, ys, color='black', linewidth=1.5)
    ax.text(0.02, 0.98, eq, transform=ax.transAxes, va='top', ha='left', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.75, edgecolor='lightgray'))
    return eq


# ---------------------------------------------------------------------------
# Load the CAMELS tables (all indexed by the catchment identifier gauge_id)
# ---------------------------------------------------------------------------
hydro = pd.read_csv('data/camels_hydro.txt', sep=';')
geol  = pd.read_csv('data/camels_geol.txt', sep=';')
clim  = pd.read_csv('data/camels_clim.txt', sep=';')
soil  = pd.read_csv('data/camels_soil.txt', sep=';')
vege  = pd.read_csv('data/camels_vege.txt', sep=';')
topo  = pd.read_csv('data/camels_topo.txt', sep=';')
name  = pd.read_csv('data/camels_name.txt', sep=';')

# Merge everything into one wide table, one row per catchment
merged = (hydro
          .merge(topo, on='gauge_id', how='left')
          .merge(soil, on='gauge_id', how='left')
          .merge(clim, on='gauge_id', how='left')
          .merge(vege, on='gauge_id', how='left')
          .merge(geol, on='gauge_id', how='left')
          .merge(name[['gauge_id', 'huc_02']], on='gauge_id', how='left')
          .set_index('gauge_id'))

# HUC-02 is a 2-digit code for one of 18 major US hydrologic regions — used below to filter the
# map, the same way `gauge_country` filters the EStreams map. It's not a continuous variable, so
# it's kept out of the histogram/scatter/correlation variable lists.
merged['huc_02'] = merged['huc_02'].apply(lambda v: f'{int(v):02d}' if pd.notna(v) else v)
numeric_vars = merged.select_dtypes(include=[np.number]).columns.tolist()

# Catchment-boundary basemap (bundled locally — no internet needed, unlike the EStreams map)
states = gpd.read_file('shapefiles/usa-states-census-2014.shp')

print(f'{merged.shape[0]:,} catchments, {merged.shape[1]} attributes '
      f'({len(numeric_vars)} numeric), {merged["huc_02"].nunique()} HUC-02 regions')
merged.head()


In [ ]:
def run_histogram(variable='runoff_ratio', bins=40, xmin='', xmax='', ymin='', ymax=''):
    values = merged[variable].dropna()
    fig, ax = plt.subplots()
    ax.hist(values, bins=bins, color='#3E7CB1', edgecolor='white')
    ax.set_xlabel(variable)
    ax.set_ylabel('number of catchments')
    ax.set_title(f'Distribution of {variable} across {len(values):,} US catchments')
    _apply_limits(ax, xmin, xmax, ymin, ymax)
    plt.show()

    print(values.describe().to_string())
    print(f'skewness: {values.skew():.2f}')

interact(run_histogram,
         variable=widgets.Dropdown(options=numeric_vars, value='runoff_ratio',
                                    description='variable:'),
         bins=widgets.IntSlider(min=5, max=100, step=5, value=40, description='bins:'),
         xmin=widgets.Text(value='', description='x min:', placeholder='auto'),
         xmax=widgets.Text(value='', description='x max:', placeholder='auto'),
         ymin=widgets.Text(value='', description='y min:', placeholder='auto'),
         ymax=widgets.Text(value='', description='y max:', placeholder='auto'));


### Questions

**Q1.** What is a typical (e.g. median) runoff ratio for these US catchments? How does it compare
to the value you found for European catchments in the EStreams companion notebook?

**Q2.** A few catchments here have a runoff ratio slightly above 1 (i.e. more water leaves the
catchment than falls on it, on average). Physically, how is that possible?

**Q3.** The scatter tool below reports both a *Spearman rank correlation coefficient* and a
*Pearson correlation coefficient*. What is the conceptual difference between the two?

**Q4.** If you want to quantify the **strength** of a relationship, do you look at the correlation
coefficient or the p-value? What about the **significance** of a relationship? Why are these two
different questions?

In [ ]:
def run_scatter(x='aridity', y='runoff_ratio', color_by='none', fit_type='linear',
                 logx=False, logy=False,
                 xmin='', xmax='', ymin='', ymax='', cmin='', cmax=''):
    # de-duplicate: color_by may legitimately be the same variable as x or y
    cols = list(dict.fromkeys([x, y] + ([color_by] if color_by != 'none' else [])))
    data = merged[cols].dropna()
    if logx:
        data = data[data[x] > 0]
    if logy:
        data = data[data[y] > 0]

    fig, ax = plt.subplots()
    if color_by == 'none':
        ax.scatter(data[x], data[y], s=10, alpha=0.5, color='#3E7CB1', edgecolor='none')
    else:
        cvmin, cvmax = _parse_float(cmin), _parse_float(cmax)
        sca = ax.scatter(data[x], data[y], c=data[color_by], cmap='viridis',
                          vmin=cvmin, vmax=cvmax, s=14, alpha=0.75, edgecolor='none')
        plt.colorbar(sca, ax=ax, label=color_by, shrink=0.85)

    eq = _fit_and_label(ax, data[x], data[y], fit_type)

    if logx:
        ax.set_xscale('log')
    if logy:
        ax.set_yscale('log')

    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f'n = {len(data):,} catchments')
    _apply_limits(ax, xmin, xmax, ymin, ymax)
    plt.show()

    rho, p_s = stats.spearmanr(data[x], data[y])
    r, p_p = stats.pearsonr(data[x], data[y])
    print(f'Spearman r = {rho:.3f}  (p = {p_s:.1e})')
    print(f'Pearson  r = {r:.3f}  (p = {p_p:.1e})')
    if eq:
        print(f'fit ({fit_type}): {eq}')

interact(run_scatter,
         x=widgets.Dropdown(options=numeric_vars, value='aridity', description='x:'),
         y=widgets.Dropdown(options=numeric_vars, value='runoff_ratio', description='y:'),
         color_by=widgets.Dropdown(options=['none'] + numeric_vars, value='none',
                                    description='color by:'),
         fit_type=widgets.Dropdown(
             options=['none', 'linear', 'quadratic', 'cubic', 'exponential', 'logarithmic', 'power'],
             value='linear', description='fit:'),
         logx=widgets.Checkbox(value=False, description='log-scale x'),
         logy=widgets.Checkbox(value=False, description='log-scale y'),
         xmin=widgets.Text(value='', description='x min:', placeholder='auto'),
         xmax=widgets.Text(value='', description='x max:', placeholder='auto'),
         ymin=widgets.Text(value='', description='y min:', placeholder='auto'),
         ymax=widgets.Text(value='', description='y max:', placeholder='auto'),
         cmin=widgets.Text(value='', description='color min:', placeholder='auto'),
         cmax=widgets.Text(value='', description='color max:', placeholder='auto'));


**Q5.** Using the scatter tool, explore which catchment characteristics relate most strongly to
`runoff_ratio` — try `aridity`, `p_mean`, `frac_snow`, `elev_mean`, `slope_mean`, `soil_porosity`,
`baseflow_index`, and a few of your own choosing. Which single variable explains the most
variance? Does this match what the Budyko framework predicts (that aridity should be the dominant
control)? Now compare with the EStreams companion notebook — do the same variables come out on top
for European catchments, or does the ranking change?

In [ ]:
USA_XLIM = (-127, -67)
USA_YLIM = (24, 50)

regions = ['all'] + sorted(merged['huc_02'].dropna().unique().tolist())

def map_maker(variable='runoff_ratio', region='all',
              cmin='', cmax='', xmin='', xmax='', ymin='', ymax=''):
    subset = merged if region == 'all' else merged[merged['huc_02'] == region]
    data = subset[['gauge_lon', 'gauge_lat', variable]].dropna()

    fig, ax = plt.subplots(figsize=(13, 9))
    states.boundary.plot(ax=ax, color='black', linewidth=0.7, zorder=0)

    vmin = _parse_float(cmin)
    vmax = _parse_float(cmax)
    if vmin is None:
        vmin = data[variable].quantile(0.05)
    if vmax is None:
        vmax = data[variable].quantile(0.95)

    sca = ax.scatter(data['gauge_lon'], data['gauge_lat'], c=data[variable], cmap='viridis',
                      vmin=vmin, vmax=vmax, s=14, zorder=1)
    # a smaller, thinner colorbar so it doesn't dominate the (now larger) map
    plt.colorbar(sca, ax=ax, label=variable, shrink=0.4, fraction=0.035, pad=0.02)
    ax.set_xlabel('longitude')
    ax.set_ylabel('latitude')
    ax.set_aspect('equal')

    if region == 'all':
        ax.set_xlim(*USA_XLIM)
        ax.set_ylim(*USA_YLIM)
    else:
        pad = 1.0
        ax.set_xlim(data['gauge_lon'].min() - pad, data['gauge_lon'].max() + pad)
        ax.set_ylim(data['gauge_lat'].min() - pad, data['gauge_lat'].max() + pad)

    # manual overrides (lon/lat limits), applied last so they always win
    _apply_limits(ax, xmin, xmax, ymin, ymax)

    ax.set_title(f'{variable} — {len(data):,} catchments'
                 + ('' if region == 'all' else f' (HUC-02 region {region})'))
    plt.show()

interact(map_maker,
         variable=widgets.Dropdown(options=numeric_vars, value='runoff_ratio',
                                    description='variable:'),
         region=widgets.Dropdown(options=regions, value='all', description='HUC-02 region:'),
         cmin=widgets.Text(value='', description='color min:', placeholder='auto (p5)'),
         cmax=widgets.Text(value='', description='color max:', placeholder='auto (p95)'),
         xmin=widgets.Text(value='', description='lon min:', placeholder='auto'),
         xmax=widgets.Text(value='', description='lon max:', placeholder='auto'),
         ymin=widgets.Text(value='', description='lat min:', placeholder='auto'),
         ymax=widgets.Text(value='', description='lat max:', placeholder='auto'));


In [ ]:
def correlation_overview(method='spearman', cmin='', cmax=''):
    corr = merged[numeric_vars].corr(method=method)
    vmin = _parse_float(cmin)
    vmax = _parse_float(cmax)
    if vmin is None:
        vmin = -1
    if vmax is None:
        vmax = 1
    fig, ax = plt.subplots(figsize=(15, 13))
    sns.heatmap(corr, cmap='BrBG', linewidths=0.1, vmin=vmin, vmax=vmax, ax=ax,
                cbar_kws=dict(shrink=0.7))
    ax.set_title(f'{method.capitalize()} correlation between all CAMELS attributes')
    plt.show()

interact(correlation_overview,
         method=widgets.Dropdown(options=['pearson', 'spearman'], value='spearman',
                                  description='method:'),
         cmin=widgets.Text(value='', description='color min:', placeholder='auto (-1)'),
         cmax=widgets.Text(value='', description='color max:', placeholder='auto (1)'));

# Note: unlike the EStreams companion notebook (which curates ~15 core variables out of ~180),
# CAMELS is small enough that the full attribute set (~40 numeric variables) is shown directly.


### Predicting a signature with machine learning

Every tool above looks at *one* predictor at a time. Catchment behaviour is usually controlled by
several factors acting together, so this last tool takes a different approach: a **random forest
regressor** (an ensemble of decision trees) learns to predict a chosen hydrologic signature from
up to three catchment attributes you pick, trained on all 671 CAMELS catchments.

Pick a signature to predict and one to three predictors, optionally type in values for those
predictors (leave a box blank to use that predictor's dataset median), and the tool will:

1. hold out a fraction of catchments (adjustable) to report how well the model generalizes to
   catchments it hasn't seen (R2 and MAE on that held-out test set),
2. show which of your chosen predictors the forest actually leans on (feature importance), and
3. predict the signature for the input you typed, using a forest trained on **all** catchments.

**Q6.** Pick `runoff_ratio` as the signature and `aridity` as the only predictor. What test R2 do
you get? Now add `p_mean` and `frac_snow` as predictors 2 and 3 &mdash; does R2 improve, and which
variable gets the highest feature importance? Does that match what you found with the scatter and
correlation tools above?

**Q7.** Try predicting a signature that climate alone explains poorly, e.g. `baseflow_index` or
`slope_fdc`, using the same three predictors. Does the random forest still fit well? What does a
low test R2 tell you that a low correlation coefficient in the scatter tool does not (and vice
versa)?

**Q8.** Enter an `aridity` value well outside the range you see in the map or histogram tools
above (e.g. `5`, far more arid than any CAMELS catchment) and predict `runoff_ratio`. Does the
prediction still make physical sense? Why are random forests generally unreliable when
*extrapolating* beyond the range of their training data, unlike a fitted linear or power-law
equation from the scatter tool?

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

def rf_predictor(signature='runoff_ratio', predictor1='aridity', predictor2='p_mean',
                  predictor3='none', val1='', val2='', val3='',
                  n_estimators=200, test_size=0.2, random_state=0):
    # collect the (up to 3) distinct predictors chosen, dropping 'none' and the signature itself
    chosen = [(predictor1, val1), (predictor2, val2), (predictor3, val3)]
    predictors, entered = [], {}
    for var, val in chosen:
        if var == 'none' or var == signature or var in predictors:
            continue
        predictors.append(var)
        entered[var] = val
    if not predictors:
        print('Choose at least one predictor that is different from the signature.')
        return

    data = merged[predictors + [signature]].dropna()
    X, y = data[predictors], data[signature]

    # train/test split, purely to report how well the forest generalizes
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state)
    rf = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)
    rf.fit(X_train, y_train)
    y_pred_test = rf.predict(X_test)
    r2 = r2_score(y_test, y_pred_test)
    mae = mean_absolute_error(y_test, y_pred_test)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
    ax1.plot(lims, lims, color='gray', linewidth=1, linestyle='--')
    ax1.scatter(y_test, y_pred_test, s=14, alpha=0.6, color='#3E7CB1', edgecolor='none')
    ax1.set_xlabel(f'observed {signature}')
    ax1.set_ylabel(f'predicted {signature}')
    ax1.set_title(f'held-out test set (n = {len(y_test):,})')

    importances = pd.Series(rf.feature_importances_, index=predictors).sort_values()
    ax2.barh(importances.index, importances.values, color='#3E7CB1')
    ax2.set_xlabel('feature importance')
    ax2.set_title('which predictors does the forest use?')
    plt.tight_layout()
    plt.show()

    print(f'Trained on {len(X_train):,} catchments, tested on {len(X_test):,} '
          f'({test_size:.0%} held out)')
    print(f'Test R2  = {r2:.3f}')
    print(f'Test MAE = {mae:.3g}')

    # refit on every available catchment for the "operational" prediction below
    rf_full = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)
    rf_full.fit(X, y)

    inputs = {}
    for var in predictors:
        parsed = _parse_float(entered[var])
        inputs[var] = parsed if parsed is not None else X[var].median()
    input_row = pd.DataFrame([inputs])[predictors]
    prediction = rf_full.predict(input_row)[0]

    print()
    print('Prediction for your chosen catchment:')
    for var in predictors:
        auto = '' if _parse_float(entered[var]) is not None else '  (auto: dataset median, box left blank)'
        print(f'  {var:>18s} = {inputs[var]:.4g}{auto}')
    print(f'  {"predicted " + signature:>18s} = {prediction:.4g}')

interact(rf_predictor,
         signature=widgets.Dropdown(options=numeric_vars, value='runoff_ratio',
                                     description='signature:'),
         predictor1=widgets.Dropdown(options=numeric_vars, value='aridity',
                                      description='predictor 1:'),
         predictor2=widgets.Dropdown(options=['none'] + numeric_vars, value='p_mean',
                                      description='predictor 2:'),
         predictor3=widgets.Dropdown(options=['none'] + numeric_vars, value='none',
                                      description='predictor 3:'),
         val1=widgets.Text(value='', description='value 1:', placeholder='auto (median)'),
         val2=widgets.Text(value='', description='value 2:', placeholder='auto (median)'),
         val3=widgets.Text(value='', description='value 3:', placeholder='auto (median)'),
         n_estimators=widgets.IntSlider(min=50, max=500, step=50, value=200,
                                         description='n trees:'),
         test_size=widgets.FloatSlider(min=0.1, max=0.5, step=0.05, value=0.2,
                                        description='test frac:'));

### References

Addor, N., Newman, A. J., Mizukami, N., & Clark, M. P. (2017). The CAMELS data set: catchment
attributes and meteorology for large-sample studies. *Hydrology and Earth System Sciences*,
21(10), 5293–5313. https://doi.org/10.5194/hess-21-5293-2017

Newman, A. J., Clark, M. P., Sampson, K., Wood, A., Hay, L. E., Bock, A., Viger, R. J., Blodgett,
D., Brekke, L., Arnold, J. R., Hopson, T., & Duan, Q. (2015). Development of a large-sample
watershed-scale hydrometeorological data set for the contiguous USA: data set characteristics and
assessment of regional variability in hydrologic model performance. *Hydrology and Earth System
Sciences*, 19, 209–223. https://doi.org/10.5194/hess-19-209-2015

Compare with the European example: [EStreams_Lecture_RunoffRatio](https://github.com/wberghuijs/EStreams_Lecture_RunoffRatio)
(Nascimento et al., 2024 — EStreams dataset, 17,130 catchments across 41 European countries).